In [ ]:
# notebook utilities
%load_ext autoreload
%autoreload 2

# standard library imports
from datetime import datetime
start_time = datetime.now()
print(f"Start time: {start_time}")

In [ ]:
%%time

# Standard library imports
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
import geopandas as gpd

# Import external functions
from src.compute_distances import compute_distances

# Import some functions from Fleur's master thesis research
from src.Fleur.network_gpbp import get_nodes_and_edges
from src.Fleur.InputDataPreprocessingv1 import (
    CurrentHospitals, 
    NewHospitals, 
    NewHospitalsGrid, 
    PopulationFB, 
    NewHospitalsCSV, 
)

# We resuse some data as curated by Fleur for her master thesis research. 
data_folder = Path('./cases/Fleur/Data_vietnam')

In [ ]:
%%time
# Preprocessing of the road network
nodes, edges_attr, network = get_nodes_and_edges( data_folder / 'road_osm_preprocessed.geojson' )

In [ ]:
%%time

# Population Data
# Round the coordinates to cluster the population. 8 digits: no rounding
digits_rounding = 8

read_population = pd.read_csv( data_folder / 'pop_2020_1km.csv').reset_index()
array_household, population = PopulationFB(digits_rounding, read_population, network, nodes)

In [ ]:
%%time
# Current Hospitals -- stroke facilities in this case
health_facilities =  pd.read_csv(data_folder / 'stroke-facs.csv').reset_index()
health_facilities = health_facilities[['index','longitude','latitude','Name_English']]
current_hospitals_ID, current_hospitals = CurrentHospitals(health_facilities, network, nodes)

len(current_hospitals)

In [ ]:
instance = 'vietnam'
road_path = 'osm'
pop_path = 'facebook'
distance_threshold_largest = 300

In [ ]:
%%time 

folder = Path('generated_data_and_distances').joinpath(instance)
folder.mkdir(parents=True, exist_ok=True)
current_hospitals.to_pickle(folder.joinpath('current_hospitals').with_suffix('.pkl'))

folder = folder.joinpath(pop_path)
folder.mkdir(parents=True, exist_ok=True)
population.to_pickle(folder.joinpath(f'population').with_suffix('.pkl'))

In [ ]:
%%time

hospitals = dict()
for g in [10, 5, 1]:
    hospitals[g] = dict()
    grid = gpd.read_file(data_folder / f'potential_location_grid_{g}.geojson')
    new_hospitals_ID, hospitals[g]['new'] = NewHospitalsGrid(current_hospitals, grid, network, nodes)
    hospitals[g]['new']['Old_Cluster_ID'] = hospitals[g]['new'].Name.apply(lambda x: x.split('_')[-1][1:]).astype(np.uint64)
    hospitals[g]['all'] = pd.concat( 
        [
            current_hospitals[['Hosp_ID','nearest_node','hosp_dist_road_estrada','Latitude','Longitude']].rename(columns={"Hosp_ID": "ID"})
        ,
            hospitals[g]['new'][['Cluster_ID','nearest_node','hosp_dist_road_estrada','Latitude','Longitude']].rename(columns={"Cluster_ID": "ID"}) 
        ] 
    ).set_index('ID',drop=False)

    grid_folder = folder.joinpath(f'{g}kmGrid')
    grid_folder.mkdir(parents=True, exist_ok=True)
    hospitals[g]['all'].to_pickle(grid_folder.joinpath('all_hospitals').with_suffix('.pkl'))
    hospitals[g]['new'].to_pickle(grid_folder.joinpath('new_hospitals').with_suffix('.pkl'))

    print(f"Number of potential hospitals in {g:>2}km grid: {len(hospitals[g]['new']):>8,}")

In [ ]:
%%time

for g in hospitals.keys():
    print(f"Computing distances for {g}km grid...")
    matrix_df = compute_distances( population, hospitals[g]['all'], distance_threshold_largest, network )
    display(matrix_df.total_dist.describe().to_frame().style.format('{:,.1f}'))

    print(f"Saving distances for {g}km grid...")
    grid_folder = folder.joinpath(f'{g}kmGrid')
    matrix_file_name = grid_folder.joinpath(
        f'distances_{road_path}_max_{distance_threshold_largest}km'
    ).with_suffix('.pkl')
    matrix_df.sort_values(['pop_id','hosp_id']).to_pickle(matrix_file_name)

In [ ]:
end_time = datetime.now()
print(f"End time: {end_time}")

# Calculate elapsed time
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time}")